# Does the ownship's own velocity uncertainty belong in $\Sigma_{V_{rel}}$?

`ProbabilisticFTR` replaces each deterministic FTR separation check with an exceedance
probability $P(\lVert d_{CPA}\rVert > R_{PZ})$ and recovers only when both criteria clear a
confidence threshold $\gamma$. That probability marginalises position uncertainty in closed form
and velocity uncertainty over the direction of $V_{rel}$, so **what goes into
$\Sigma_{V_{rel}}$ sets how wide the angular spread is** — and therefore how long the aircraft
holds its avoidance manoeuvre.

There are two defensible answers and they are not equivalent:

- **$\Sigma_{V_{rel}} = \Sigma_{V_o} + \Sigma_{V_i}$** — the independent-errors sum, the symmetric
  counterpart of $\Sigma_{rel} = \Sigma_o + \Sigma_i$ on the position side.
- **$\Sigma_{V_{rel}} = \Sigma_{V_i}$** — the ownship's side of *both* FTR criteria is
  `own.desired`, its own declared intent. An aircraft does not misread the velocity it is about to
  fly, so on this reading $\Sigma_{V_o}$ describes a *current* velocity neither criterion consults.

This notebook measures the cost of that choice against `PastCPA` as a reference, across crossing
angle. Both are run at $\gamma = 0.999$.

## What is measured, and with which estimator

Two dependent variables, and they need **different estimators** — this is the one methodological
point worth being careful about here.

- **$P(\mathrm{LoS})$** is a rare-event probability once the recovery logic is doing its job, so it
  is estimated by the fixed-effort **IPS** splitting estimator (`opencdarr.parallel`), which
  concentrates particles on the trajectories heading toward the rare set.
- **Median achieved separation** is a *bulk* statistic, and IPS cannot supply it. Its cloud is
  deliberately resampled toward the rare set at every shell, so the min-separations it carries are
  a conditioned sample, not a sample of the encounter distribution — taking their median would
  report a number biased low by construction. It comes from a plain **Monte Carlo** pass instead.

The two therefore run on the same geometry but through separate estimators, which is why the cell
below defines the scenario once and both passes read it.

In [1]:
import os

# Keep this notebook to four cores: joblib workers below, and the BLAS/OpenMP pools
# numpy would otherwise size to the whole machine (each worker spawning its own).
N_JOBS = 4
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed

from opencdarr import cache
from opencdarr.cd import StateBased
from opencdarr.cns.navigation import GnssNavigation
from opencdarr.config import (
    Config, ConflictConfig, MethodsConfig, ScenarioConfig, SimulationConfig,
)
from opencdarr.cr import MVP
from opencdarr.crr import PastCPA, ProbabilisticFTR
from opencdarr.estimator import combine_ipr, estimate_ipr
from opencdarr.fleet import Agent, build_env
from opencdarr.ips import Particle
from opencdarr.parallel import estimate_rare_prob
from opencdarr.performance import M600
from opencdarr.rng import children, root_seed_sequence
from opencdarr.scenario import create_conflict
from opencdarr.state import AircraftState

## The scenario

Two M600s at **20 kts** (10.2889 m/s), `MVP` resolving with a 1.05 margin, `StateBased` detection
at a 120 s lookahead, `rpz = 50 m`. GNSS self-fix error is **10 m / 1 m/s** (95% radial) on each
aircraft — the level Experiment 3 of the paper holds fixed. Intent is **not** shared, so criterion
2 reads the onset velocity `SeparationManager` records for the pair at first detection.

The miss distance is pinned at `dcpa = 0` — a nominal dead-on collision — so the recovery
criterion is the only thing standing between the pair and a loss of separation, and the crossing
angle is the only thing that varies. Encounters are spawned at `tlos = 180 s`, which is
$1.5\times$ the lookahead: the paper's spawn rule, deliberately including geometries where noise
can trigger detection before the nominal time to intrusion.

**A consequence of the small angles worth stating up front.** At a 2° crossing the closing speed is
only $2 \cdot 10.29 \cdot \sin 1° \approx 0.36$ m/s, so spawning 180 s from intrusion puts the
intruder just **115 m** away — barely twice the protected zone, and it stays there for minutes.
That is not an artefact of the setup; it *is* what a 2° crossing looks like, and it is exactly the
regime the probabilistic criterion exists for.

In [2]:
SPEED = 10.2889      # 20 kts [m/s]
RPZ = 50.0           # protected zone [m]
T_LOOKAHEAD = 120.0  # detection lookahead [s]
TLOS = 180.0         # spawn at 1.5x the lookahead [s]
DCPA = 0.0           # pinned dead-on geometry [m]
MARGIN = 1.05        # MVP resolution-zone margin
POS_CI95 = 10.0      # 95% radial position accuracy [m]
VEL_CI95 = 1.0       # 95% radial velocity accuracy [m/s]
DT = 0.5             # integration step [s]
T_MAX = 600.0        # encounter cap [s]
GAMMA = 0.999        # probabilistic-FTR confidence threshold

ANGLES = [2.0, 5.0, 10.0, 45.0, 90.0, 180.0]

# The three recovery criteria under comparison. The first two differ in exactly one thing:
# whether the ownship's own declared vel_ci95 enters the relative-velocity covariance.
METHODS = {
    "prob_ftr_both": (
        r"Prob-FTR, $\Sigma_{V_o}+\Sigma_{V_i}$",
        lambda: ProbabilisticFTR(prob_threshold=GAMMA, velocity_uncertainty="both"),
    ),
    "prob_ftr_intruder": (
        r"Prob-FTR, $\Sigma_{V_i}$ only",
        lambda: ProbabilisticFTR(prob_threshold=GAMMA, velocity_uncertainty="intruder"),
    ),
    "pastcpa": ("Past-CPA", lambda: PastCPA(bouncing_guard=True)),
}
LABEL = {k: v[0] for k, v in METHODS.items()}

for dpsi in ANGLES:
    v_rel = 2.0 * SPEED * math.sin(math.radians(dpsi) / 2.0)
    rng0 = TLOS * v_rel + RPZ  # create_conflict's initial relative range
    print(f"dpsi = {dpsi:5.0f} deg   closing speed = {v_rel:6.2f} m/s   initial range = {rng0:7.0f} m")

dpsi =     2 deg   closing speed =   0.36 m/s   initial range =     115 m
dpsi =     5 deg   closing speed =   0.90 m/s   initial range =     212 m
dpsi =    10 deg   closing speed =   1.79 m/s   initial range =     373 m
dpsi =    45 deg   closing speed =   7.87 m/s   initial range =    1467 m
dpsi =    90 deg   closing speed =  14.55 m/s   initial range =    2669 m
dpsi =   180 deg   closing speed =  20.58 m/s   initial range =    3754 m


## Median achieved separation — plain Monte Carlo

300 encounters per cell, geometry pinned (`dpsi`, `dcpa`) so the only randomness is the CNS noise
each aircraft draws. Chunked across cores over contiguous slices of the *same* seed fan-out and
pooled with `combine_ipr`, which is bit-identical to the serial run — offsetting the seed per
chunk would not be.

In [3]:
N_ENCOUNTERS = 300
MC_SEED = 20260803
CACHE_DIR = Path(".opencdarr_cache")


def mc_config() -> Config:
    """The plain-MC run configuration. ``methods.recovery`` is unused — ``estimate_ipr`` takes the
    criterion object directly, which is how ``ProbabilisticFTR`` (deliberately absent from the
    string registry) is reachable at all."""
    return Config(
        seed=MC_SEED,
        n_encounters=N_ENCOUNTERS,
        scenario=ScenarioConfig(
            aircraft_type="M600", speed=SPEED, dcpa_max=RPZ, tlos=TLOS,
            pos_ci95=POS_CI95, vel_ci95=VEL_CI95,
        ),
        conflict=ConflictConfig(rpz=RPZ, t_lookahead=T_LOOKAHEAD),
        methods=MethodsConfig(detection="statebased", resolution="mvp", recovery=None,
                              margin=MARGIN, bouncing_guard=True),
        simulation=SimulationConfig(dt=DT, t_max=T_MAX, done_timeout=10.0),
    )


def _mc_slice(make_recovery, dpsi: float, lo: int, hi: int):
    """One contiguous slice of the encounter fan-out (a separate process builds its own criterion)."""
    seqs = children(root_seed_sequence(MC_SEED), lo, hi)
    return estimate_ipr(
        mc_config(), M600, StateBased(), MVP(MARGIN), make_recovery(),
        navigation=GnssNavigation(), dpsi=dpsi, dcpa=DCPA, seqs=seqs,
    )


def run_mc(method: str, dpsi: float, n_jobs: int = N_JOBS, chunks: int = 8):
    """Median achieved separation and P(LoS) from the plain-MC pass, cached on disk."""
    params = {
        "estimator": "mc", "method": method, "dpsi": dpsi, "dcpa": DCPA, "tlos": TLOS,
        "speed": SPEED, "rpz": RPZ, "t_lookahead": T_LOOKAHEAD, "margin": MARGIN,
        "pos_ci95": POS_CI95, "vel_ci95": VEL_CI95, "gamma": GAMMA,
        "dt": DT, "t_max": T_MAX, "n_encounters": N_ENCOUNTERS,
    }
    key = cache.run_key(params, seed=MC_SEED)
    make_recovery = METHODS[method][1]

    def compute():
        edges = np.linspace(0, N_ENCOUNTERS, chunks + 1).astype(int)
        parts = Parallel(n_jobs=n_jobs)(
            delayed(_mc_slice)(make_recovery, dpsi, int(a), int(b))
            for a, b in zip(edges[:-1], edges[1:]) if b > a
        )
        return combine_ipr(parts)

    return cache.load_or_run(key, compute)


mc = {}
t0 = time.perf_counter()
for method in METHODS:
    for dpsi in ANGLES:
        mc[method, dpsi] = run_mc(method, dpsi)
        r = mc[method, dpsi]
        print(f"{method:18s} dpsi={dpsi:5.0f}  median sep = {r.median_min_sep:7.1f} m   "
              f"P(LoS)_mc = {r.p_los:.3f}  ({r.n_encounters} encounters)")
print(f"\nMonte Carlo pass: {time.perf_counter() - t0:.0f} s")

prob_ftr_both      dpsi=    2  median sep =    98.6 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_both      dpsi=    5  median sep =   145.4 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_both      dpsi=   10  median sep =   207.0 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_both      dpsi=   45  median sep =   154.8 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_both      dpsi=   90  median sep =   146.3 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_both      dpsi=  180  median sep =   150.4 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_intruder  dpsi=    2  median sep =    98.6 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_intruder  dpsi=    5  median sep =   141.8 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_intruder  dpsi=   10  median sep =   189.9 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_intruder  dpsi=   45  median sep =   134.2 m   P(LoS)_mc = 0.000  (300 encounters)
prob_ftr_intruder  dpsi=   90  median sep =   130.2 m   P(LoS)_mc = 0.000  (300 

## $P(\mathrm{LoS})$ — IPS

One shell ladder shared by every cell, ending at the 50 m event. A shared ladder is not required —
$\hat P = \prod_k S_k/N$ is unbiased for the terminal shell whatever the intermediate shells are —
but with three methods to overlay it lines the survival curves up on identical $x$ values, and the
cost of a shell whose survival is $\approx 1$ is one pass over $N$ particles.

The top shell sits at 100 m so that even the 2° geometry, which starts only 115 m out, begins above
it. Steps are 1 m through the band just above the protected zone, where the cliff is, and coarser
elsewhere.

**This pass is slow, and it is the shallow angles that make it so.** The 2° and 5° cells run the full
`t_max` because the pair closes at well under 1 m/s, and every decision of every particle at every
shell evaluates the $K_\theta = 256$ quadrature twice. Two things were checked before settling on
these settings:

- **`dt` does not matter.** At `dt = 0.5` and `dt = 1.0` the 2° estimate is identical to four
  significant figures (1.736e-05). The multirotor reaches its commanded velocity essentially
  instantly and commands are issued at 1 Hz, so halving the integration step changes nothing but
  the number of integration calls. The cost is in the CDR decisions, not the integration.
- **`t_max` does.** Shortening it to 300 s leaves the probabilistic estimates untouched but moves
  Past-CPA at 5° by 10% (7.5e-03 to 8.3e-03) — that criterion holds until past CPA, and at a 5°
  crossing some particles are still reaching the deep shells after 300 s. So the tail is not
  discardable and `t_max` stays at 600 s.

Three replications is enough to see the shape and to catch a cell that collapses, but a
3-replication log-space CI is **indicative only** — read the intervals as rough. Particles were
kept at 200 rather than replications at 5 because the particle count is what stops the shallow-angle
cells collapsing to a bound.

Everything runs on **four cores** (`N_JOBS`), with the BLAS/OpenMP thread pools pinned to 1 so
the workers do not each spawn their own. Every cell is cached on disk, so a re-run costs nothing
and an interrupted one resumes.

In [4]:
N_PARTICLES = 200
REPS = 3
IPS_SEED = 20260803
EVENT = RPZ

LADDER = [100.0, 90.0, 80.0, 72.0, 66.0, 62.0, 59.0, 57.0,
          56.0, 55.0, 54.0, 53.0, 52.0, 51.0, EVENT]
print(f"{len(LADDER)} shells: {LADDER}")


def make_start(method: str, dpsi: float) -> Particle:
    """The single starting particle: geometry is pinned, so every particle begins identically and
    all the spread comes from the forward CNS noise IPS splits on."""
    own = AircraftState(id="OWN", lat=52.0, lon=4.0, trk=0.0, gs=SPEED,
                        pos_ci95=POS_CI95, vel_ci95=VEL_CI95)
    intr = create_conflict(own, intr_id="INT", dpsi=dpsi, dcpa=DCPA,
                           tlos=TLOS, rpz=RPZ, side=1)
    agents = [Agent(own, M600), Agent(intr, M600)]
    env = build_env(agents, rpz=RPZ, t_lookahead=T_LOOKAHEAD, dt=DT,
                    detector=StateBased(), resolver=MVP(MARGIN),
                    recovery=METHODS[method][1](), navigation=GnssNavigation(),
                    t_max=T_MAX, done_timeout=10.0)
    return Particle(env=env, state=env.initial_state(agents))


def run_ips(method: str, dpsi: float):
    """The IPS estimate for one cell, read from disk when this exact run is already there."""
    params = {
        "estimator": "ips", "method": method, "dpsi": dpsi, "dcpa": DCPA, "tlos": TLOS,
        "speed": SPEED, "rpz": RPZ, "t_lookahead": T_LOOKAHEAD, "margin": MARGIN,
        "pos_ci95": POS_CI95, "vel_ci95": VEL_CI95, "gamma": GAMMA,
        "dt": DT, "t_max": T_MAX, "levels": list(LADDER),
        "n_particles": N_PARTICLES, "reps": REPS,
    }
    key = cache.run_key(params, seed=IPS_SEED)
    start = make_start(method, dpsi)
    return cache.load_or_run(key, lambda: estimate_rare_prob(
        lambda seq, s=start: s, LADDER,
        n_particles=N_PARTICLES, reps=REPS, seed=IPS_SEED, n_jobs=N_JOBS))


ips = {}
t0 = time.perf_counter()
for method in METHODS:
    for dpsi in ANGLES:
        est = ips[method, dpsi] = run_ips(method, dpsi)
        tag = "" if est.n_collapsed == 0 else f"  [{est.n_collapsed}/{REPS} collapsed]"
        print(f"{method:18s} dpsi={dpsi:5.0f}  P(LoS) = {est.prob:.3e}{tag}")
print(f"\nIPS pass: {time.perf_counter() - t0:.0f} s")

15 shells: [100.0, 90.0, 80.0, 72.0, 66.0, 62.0, 59.0, 57.0, 56.0, 55.0, 54.0, 53.0, 52.0, 51.0, 50.0]


prob_ftr_both      dpsi=    2  P(LoS) = 4.884e-05


prob_ftr_both      dpsi=    5  P(LoS) = 5.729e-05


prob_ftr_both      dpsi=   10  P(LoS) = 2.731e-05


prob_ftr_both      dpsi=   45  P(LoS) = 9.752e-07


prob_ftr_both      dpsi=   90  P(LoS) = 0.000e+00  [3/3 collapsed]


prob_ftr_both      dpsi=  180  P(LoS) = 0.000e+00  [3/3 collapsed]


prob_ftr_intruder  dpsi=    2  P(LoS) = 2.702e-05


KeyboardInterrupt: 

### Collapsed replications are a bound, not a zero

A replication whose cloud runs out of survivors before the last shell reports $\hat P = 0$. That is
the estimator saying *it never got there with 200 particles*, not that the probability is zero. For
those cells the honest reading is an upper bound: at the shell where zero of $N$ survivors crossed,
the conditional crossing probability is below $3/N$ at 95% confidence (the rule of three), and
since the running minimum is monotone everything below that shell inherits the bound.

`point_estimate` below returns the estimate when the ladder was reached and that bound when it was
not, flagging which is which so the plot can mark them differently.

In [ ]:
def curve(est):
    """P(min sep < d) at each shell, averaged over the replications that reached it."""
    dists, probs = [], []
    for k, d in enumerate(est.reps[0].levels):
        reached = [r for r in est.reps if k < len(r.survival) and r.survival[k] > 0.0]
        if not reached:
            break
        dists.append(d)
        probs.append(sum(math.prod(r.survival[: k + 1]) for r in reached) / len(reached))
    return dists, probs


def point_estimate(est) -> tuple[float, bool]:
    """(value, is_bound) — the estimate, or the rule-of-three bound if every replication collapsed."""
    if est.n_collapsed < REPS:
        return est.prob, False
    dists, probs = curve(est)
    if not dists:
        return 3.0 / N_PARTICLES, True
    return probs[-1] * 3.0 / N_PARTICLES, True


summary = {}
for method in METHODS:
    for dpsi in ANGLES:
        p, bounded = point_estimate(ips[method, dpsi])
        summary[method, dpsi] = (p, bounded, mc[method, dpsi].median_min_sep)

hdr = f"{'dpsi':>6} " + " ".join(f"{m:>26}" for m in METHODS)
print(hdr)
print("-" * len(hdr))
for dpsi in ANGLES:
    row = f"{dpsi:6.0f} "
    for method in METHODS:
        p, bounded, med = summary[method, dpsi]
        row += f"  {'<' if bounded else ' '}{p:.2e} / {med:6.1f} m   "
    print(row)
print("\n'<' marks a rule-of-three upper bound (every replication collapsed); "
      "second number is the median achieved separation.")

## Results

In [ ]:
COLOUR = {"prob_ftr_both": "tab:blue", "prob_ftr_intruder": "tab:orange", "pastcpa": "tab:green"}

fig, (ax_p, ax_d) = plt.subplots(1, 2, figsize=(10.5, 4.2))

for method in METHODS:
    ps = [summary[method, a][0] for a in ANGLES]
    bounded = [summary[method, a][1] for a in ANGLES]
    meds = [summary[method, a][2] for a in ANGLES]
    c = COLOUR[method]
    ax_p.plot(ANGLES, ps, "-o", ms=4, color=c, label=LABEL[method])
    # open downward triangles mark cells that are upper bounds rather than measurements
    for a, p, b in zip(ANGLES, ps, bounded):
        if b:
            ax_p.plot(a, p, "v", ms=9, mfc="none", color=c)
    ax_d.plot(ANGLES, meds, "-o", ms=4, color=c, label=LABEL[method])

ax_d.axhline(RPZ, color="0.6", lw=0.8, ls=":")
ax_d.annotate(f"$R_{{PZ}}$ = {RPZ:.0f} m", xy=(ANGLES[0], RPZ), xytext=(0, 5),
              textcoords="offset points", color="0.4", fontsize=9)

for ax in (ax_p, ax_d):
    ax.set_xscale("log")
    ax.set_xticks(ANGLES)
    ax.set_xticklabels([f"{a:.0f}" for a in ANGLES])
    ax.set_xlabel("crossing angle [deg]")

ax_p.set_yscale("log")
ax_p.set_ylabel(r"$P(\mathrm{LoS})$")
ax_p.set_title("Loss of separation (IPS)")
ax_d.set_ylabel("median achieved separation [m]")
ax_d.set_title("Achieved separation (Monte Carlo)")
ax_p.legend(fontsize=9)

fig.tight_layout()
fig.savefig("prob_ftr_velocity_covariance.png", dpi=150, bbox_inches="tight")
plt.show()

Left: $P(\mathrm{LoS})$ against crossing angle, log–log. Open triangles mark cells where every IPS
replication ran out of particles, so the plotted value is a rule-of-three upper bound and the true
probability lies somewhere below it. Right: median achieved separation from the Monte Carlo pass,
with the protected-zone radius dotted; a curve far above it is buying its safety by holding the
avoidance manoeuvre longer and flying further off the nominal track.

Read the two panels together. A recovery criterion can always drive $P(\mathrm{LoS})$ down by
refusing to recover, and the right panel is what that costs.